# Aurora 1.5 - flash-aurora Engine

Same forecast flow as the upstream Microsoft Aurora 1.5 example (ERA5 2023-01-01, extended surface fields), using `AuroraEngine` / `DataDownloader`.

- **Deterministic preset:** `aurora_v1p5` (`AuroraV1p5`, `aurora-0.25-v1.5.ckpt`)
- **Ensemble preset:** `aurora_v1p5_ensemble` (`AuroraV1p5Ensemble`, `aurora-0.25-v1.5-ensemble.ckpt`)
- **Checkpoint / static:** HF repo `ikwessel/aurora-1.5` (ensemble uses a pinned HF revision)
- **IC:** CDS ERA5 extended surface + pressure levels, HF static pickle, computed `insolation`
- **Hourly:** pass `fine_lead_times=[1, 2, 3, 4, 5, 6]` to `rollout_stream` / `rollout_and_export`

> ERA5 is for the tutorial only. Upstream notes the model is fine-tuned on IFS, so ERA5 skill is illustrative.

> **Asset root:** default is `./assets` under the working directory. Here we use the data disk in developing environment; comment `ASSET_ROOT = Path("/root/autodl-tmp/aurora")` in the setup cell to use your own disk.

## Prerequisites

1. CDS credentials for ERA5 single-levels and pressure-levels (`CDSAPI_KEY`, `~/.cdsapirc`, or `ensure(..., prompt=True)`).
2. Checkpoint / static pickle under `ASSET_ROOT`, or Hugging Face download. If `huggingface.co` is unreachable, set `USE_HF_MIRROR = True` in the setup cell.
3. GPU VRAM similar to other 0.25 deg global Aurora tutorials.


## Set up engine


In [ ]:
from datetime import datetime
from pathlib import Path

from flash_aurora.engine import (
    AuroraEngine,
    DataDownloader,
    DEFAULT_PRESETS,
    HF_MIRROR_ENDPOINT,
    InitialConditionBuilder,
)
from flash_aurora.engine.core.redaction import safe_path

PRESET = "aurora_v1p5"
# Match upstream tutorial date: history 00:00 + 06:00 -> IC valid time 06:00.
VALID_TIME = datetime(2023, 1, 1, 6)
TIME_INDEX = 1
ROLLOUT_STEPS = 2

# Default: ./assets under the notebook working directory (created if missing).
ASSET_ROOT: Path | str | None = None

# Optional - absolute path to a mounted data disk with checkpoints/cache (uncomment to use):
ASSET_ROOT = Path("/root/autodl-tmp/aurora")

if ASSET_ROOT is not None:
    root = Path(ASSET_ROOT).expanduser()
    if not root.is_absolute():
        raise ValueError("ASSET_ROOT must be an absolute path")
    ASSET_ROOT = root.resolve()
else:
    ASSET_ROOT = (Path.cwd() / "assets").resolve()
    ASSET_ROOT.mkdir(parents=True, exist_ok=True)

USE_HF_MIRROR = False
variant = DEFAULT_PRESETS.get(PRESET).variant
local_checkpoint = ASSET_ROOT / variant.checkpoint_filename

if local_checkpoint.is_file():
    checkpoint_arg = local_checkpoint
    allow_hub_download = False
    hf_endpoint = None
    print("checkpoint:", safe_path(local_checkpoint))
else:
    checkpoint_arg = None
    allow_hub_download = True
    hf_endpoint = HF_MIRROR_ENDPOINT if USE_HF_MIRROR else None
    print("checkpoint not found locally; will download from Hugging Face")
    print("  target dir:", safe_path(ASSET_ROOT))
    print("  hf_endpoint:", hf_endpoint or "https://huggingface.co")

downloader = DataDownloader.from_preset(PRESET, asset_root=ASSET_ROOT)
engine = AuroraEngine.from_preset(
    PRESET,
    asset_root=ASSET_ROOT,
    checkpoint_path=checkpoint_arg,
    allow_hub_download=allow_hub_download,
    hf_endpoint=hf_endpoint,
)

print("source:", engine.config.source.name)
print("cache_dir:", safe_path(downloader.resolve_cache_dir()))
print("asset_root:", safe_path(ASSET_ROOT))


## 1. Download ERA5 V1p5 fields (CDS)

`DataDownloader.ensure()` retrieves the extended surface set and standard atmospheric levels into `<ASSET_ROOT>/era5_v1p5/`. Static land fields are loaded later from the HF pickle.


In [ ]:
import os
from pathlib import Path

from flash_aurora.engine.ingress.download.paths import read_cdsapirc_key

missing_files = downloader.missing(VALID_TIME)
cds_api_key = (
    os.environ.get("CDSAPI_KEY", "").strip()
    or read_cdsapirc_key()
    or None
)
has_cdsapirc = Path.home().joinpath(".cdsapirc").is_file()
if missing_files:
    print("Downloading missing ERA5 V1p5 files:", missing_files)
    result = downloader.ensure(
        VALID_TIME,
        cds_api_key=cds_api_key,
        prompt=cds_api_key is None and not has_cdsapirc,
    )
else:
    print("ERA5 V1p5 cache complete, skipping download")
    result = downloader.ensure(VALID_TIME)

print("downloaded:", result.downloaded)
print("skipped:", result.skipped)
for key, path in result.paths.items():
    print(f"  {key}: {safe_path(path)}")


## 2. Build IC, load, rollout

`CdsEra5V1p5Adapter` maps CDS short names to Aurora fields, inserts `insolation`, and attaches the 36-field static pickle. Output-only surface vars are not required in the IC.


In [ ]:
from flash_aurora.engine.ingress.adapters import IngestRequest

request = IngestRequest(
    valid_time=VALID_TIME,
    time_index=TIME_INDEX,
    cache_dir=downloader.resolve_cache_dir(),
)

batch = InitialConditionBuilder(engine.config).from_source(request)
print("surf vars:", len(batch.surf_vars), "static:", len(batch.static_vars))
print("IC time:", batch.metadata.time)

engine.load(rollout_steps=ROLLOUT_STEPS)
print("loaded model:", type(engine.model).__name__)
preds = [pred.to("cpu") for pred in engine.rollout_stream(batch, ROLLOUT_STEPS)]
print("rollout steps:", len(preds))
print("pred surf keys (sample):", sorted(preds[0].surf_vars)[:8], "...")


## 3. Hourly fine lead times

Aurora 1.5 supports sub-6h leads. Within each main AR step, all fine leads are initialised from the same previous state; only the **last** fine lead (must equal 6 h) advances the autoregressive input.

Yield count is `ROLLOUT_STEPS * len(FINE_LEAD_TIMES)`.


In [ ]:
FINE_LEAD_TIMES = [1, 2, 3, 4, 5, 6]

hourly_preds = [
    pred.to("cpu")
    for pred in engine.rollout_stream(
        batch,
        ROLLOUT_STEPS,
        fine_lead_times=FINE_LEAD_TIMES,
    )
]
print("hourly yields:", len(hourly_preds))
print("valid times:", [p.metadata.time[-1] for p in hourly_preds[:6]], "...")

# Optional NetCDF export (sequential filenames; times live in metadata):
# EXPORT_DIR = ASSET_ROOT / "output" / PRESET / "hourly"
# paths = list(engine.rollout_and_export(
#     batch,
#     ROLLOUT_STEPS,
#     export_dir=EXPORT_DIR,
#     fine_lead_times=FINE_LEAD_TIMES,
# ))
# print("wrote", len(paths), "files under", safe_path(EXPORT_DIR))


## 4. Ensemble members (optional)

Use preset `aurora_v1p5_ensemble`. Like upstream, generate members by repeating rollout and calling `reset_noise()` between members. With fine leads, keep `use_noise_accumulation=True` (Engine default).


In [ ]:
ENSEMBLE_PRESET = "aurora_v1p5_ensemble"
ENSEMBLE_MEMBERS = 2

ens_variant = DEFAULT_PRESETS.get(ENSEMBLE_PRESET).variant
ens_ckpt = ASSET_ROOT / ens_variant.checkpoint_filename
ens_checkpoint_arg = ens_ckpt if ens_ckpt.is_file() else None
ens_allow_hub = ens_checkpoint_arg is None
ens_hf_endpoint = (
    (HF_MIRROR_ENDPOINT if USE_HF_MIRROR else None) if ens_allow_hub else None
)

ensemble_engine = AuroraEngine.from_preset(
    ENSEMBLE_PRESET,
    asset_root=ASSET_ROOT,
    checkpoint_path=ens_checkpoint_arg,
    allow_hub_download=ens_allow_hub,
    hf_endpoint=ens_hf_endpoint,
)
ensemble_engine.load(rollout_steps=ROLLOUT_STEPS)
print("loaded ensemble model:", type(ensemble_engine.model).__name__)

member_forecasts = []
for member in range(ENSEMBLE_MEMBERS):
    ensemble_engine.model.reset_noise()
    member_preds = [
        pred.to("cpu")
        for pred in ensemble_engine.rollout_stream(
            batch,
            ROLLOUT_STEPS,
            fine_lead_times=FINE_LEAD_TIMES,
            use_noise_accumulation=True,
        )
    ]
    member_forecasts.append(member_preds)
    print(f"member {member}: {len(member_preds)} yields, last time {member_preds[-1].metadata.time[-1]}")
